# 23a · Visual documents and the limits of pooled crops

## The engineering question
Can image embeddings recover page content that a text transcript discards, and what layout information can they lose?

**Learning outcomes:** compare canonical-text, whole-page, and crop-vector retrieval; inspect a controlled layout ambiguity; measure representation cost; distinguish this live baseline from a trained visual-document retriever.

We create four synthetic pages locally. Two contain the same red circle and blue square in reversed vertical order. Their canonical text is deliberately identical. A query about *above* requires layout reasoning. CLIP encodes real page/crop images; Qdrant scores the crop matrices using MaxSim. The transcript baseline uses author-provided text, **not OCR**. Neither baseline is ColPali, and these pages do not reproduce ViDoRe.

ColPali trains a vision-language model to produce page multivectors and uses late interaction with query representations. Its learned page representation can preserve information unavailable to these independent CLIP crops. Testing a trained page encoder requires separately pinned weights/preprocessing, licensed page/query judgments, and another resource pilot. [ColPali paper](https://arxiv.org/abs/2407.01449v6).

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("23a", "O", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
import time
from uuid import uuid4
from PIL import Image, ImageDraw
from qdrant_client import QdrantClient, models as qm
from vector_masterclass.multimodal import clip_encode
from vector_masterclass.models import embed
from vector_masterclass.config import qdrant_url
from vector_masterclass.evidence import json_write
from vector_masterclass.late_interaction import maxsim
from vector_masterclass.retrieval import exact_search

specs = [
    ("A", ["red circle", "blue square"]),
    ("B", ["blue square", "red circle"]),
    ("C", ["green triangle", "green triangle"]),
    ("D", ["yellow square", "yellow square"]),
]
pages = []
patches = []
for name, shapes in specs:
    page = Image.new("RGB", (256, 512), "white")
    draw = ImageDraw.Draw(page)
    for row, shape in enumerate(shapes):
        y = row * 256
        if "circle" in shape:
            draw.ellipse((64, y + 64, 192, y + 192), fill="red")
        elif "triangle" in shape:
            draw.polygon([(128, y + 45), (45, y + 200), (211, y + 200)], fill="green")
        else:
            draw.rectangle((64, y + 64, 192, y + 192), fill=shape.split()[0])
    page.save(lesson.path / f"page-{name}.png")
    pages.append(page)
    patches.extend([page.crop((0, 0, 256, 256)), page.crop((0, 256, 256, 512))])
transcripts = [
    "red circle blue square",
    "red circle blue square",
    "green triangle green triangle",
    "yellow square yellow square",
]
tasks = [
    ("red circle above a blue square", "A"),
    ("blue square above a red circle", "B"),
    ("two green triangles", "C"),
    ("two yellow squares", "D"),
]
started = time.perf_counter()
whole = clip_encode(images=pages)
crops = clip_encode(images=patches).reshape(4, 2, 512)
image_seconds = time.perf_counter() - started
qimage = clip_encode(texts=[q for q, g in tasks])
text = embed(transcripts)
qtext = embed([q for q, g in tasks], role="query")
ids = [name for name, shapes in specs]
client = QdrantClient(url=qdrant_url(), timeout=30)
name = "vmc_23a_" + uuid4().hex[:8]
client.create_collection(
    name,
    vectors_config=qm.VectorParams(
        size=512,
        distance=qm.Distance.COSINE,
        multivector_config=qm.MultiVectorConfig(comparator=qm.MultiVectorComparator.MAX_SIM),
        hnsw_config=qm.HnswConfigDiff(m=0),
    ),
)
rows = []
score_error = 0.0
try:
    client.upsert(
        name,
        points=[qm.PointStruct(id=i, vector=v.tolist()) for i, v in enumerate(crops)],
        wait=True,
    )
    for (query, gold), qi, qt in zip(tasks, qimage, qtext):
        hits = client.query_points(
            name, query=[qi.tolist()], limit=4, search_params=qm.SearchParams(exact=True)
        ).points
        score_error = max(
            score_error, max(abs(h.score - maxsim(qi[None, :], crops[h.id])) for h in hits)
        )
        # Stable ID tie rule makes the layout ambiguity explicit.
        scores = {h.id: h.score for h in hits}
        crop_order = sorted(range(4), key=lambda i: (-round(scores[i], 6), ids[i]))
        rankings = {
            "canonical text": [i for i, s in exact_search(qt, text, ids, k=4)],
            "whole-page CLIP": [i for i, s in exact_search(qi, whole, ids, k=4)],
            "crop MaxSim": [ids[i] for i in crop_order],
        }
        for method, ranked in rankings.items():
            rows.append(
                {
                    "query": query,
                    "expected": gold,
                    "method": method,
                    "top": ranked[0],
                    "hit_at_1": ranked[0] == gold,
                }
            )
finally:
    client.delete_collection(name)
    client.close()
# Swapping rows cannot change MaxSim: the set of crop vectors is identical.
permutation_error = float(np.max(np.abs(crops[0] - crops[1, ::-1])))
results = lesson.table(pd.DataFrame(rows))
display(results)
costs = lesson.table(
    pd.DataFrame(
        [
            {"representation": "whole page", "raw_bytes": whole.nbytes},
            {"representation": "two crops/page", "raw_bytes": crops.nbytes},
            {"representation": "transcript", "raw_bytes": text.nbytes},
        ]
    ),
    "costs",
)
display(costs)
fig, axes = plt.subplots(1, 4, figsize=(10, 5))
for ax, page, id_ in zip(axes, pages, ids):
    ax.imshow(page)
    ax.set_title("Page " + id_)
    ax.axis("off")
fig.suptitle("A and B have identical crop sets and different layouts")
fig.tight_layout()
lesson.figure(fig)
plt.show()
json_write(
    lesson.path / "page-provenance.json",
    {
        "creator": "course deterministic PIL drawing",
        "transcripts": transcripts,
        "specifications": specs,
        "replay": False,
        "trained_page_encoder": False,
    },
)

## Interpretation and exercise
Crop MaxSim is invariant to row order. Its failure on the swapped pages is structural, even if whole-page CLIP happens to distinguish this fixture. Image retrieval returns candidate pages, not verified answers or trustworthy OCR. Small text, numeric tables, rotation, language, and pagination need independent slices. Preserve page IDs, image checksums, crop coordinates, and access policy alongside vectors; evidence extraction still needs source-grounded verification.

**Exercise:** add crop coordinates to a separate layout verifier, then rerank the two candidate pages. Compare that deterministic rule with a trained page encoder on a held-out layout set. Explain why concatenating arbitrary coordinates onto a pretrained vector changes its distance semantics. Account for page rendering, image storage, inference, token matrices, and candidate rescoring separately.

In [ ]:
manifest = lesson.finish(
    {
        "live_page_and_query_encoding": crops.shape == (4, 2, 512),
        "swapped_crop_sets_equal": permutation_error < 1e-6,
        "server_matches_MaxSim": score_error < 1e-4,
        "three_baselines": results.method.nunique() == 3,
        "layout_failure_exposed": not bool(
            results[results.method.eq("crop MaxSim")].hit_at_1.all()
        ),
    },
    observations={
        "image_encoding_seconds": image_seconds,
        "crop_permutation_error": permutation_error,
        "max_score_error": score_error,
        "corpus": "four author-created synthetic pages",
        "OCR_executed": False,
        "ColPali_reproduction": False,
        "tie_rule": "score rounded to six decimals then page ID",
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.